# NumPy, Pandas et Matplotlib — Exercices 1 a 10
## Operations matricielles, statistiques, manipulation de donnees et visualisation


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1f2e',
    'axes.edgecolor':   '#2e3448',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2e3448',
    'grid.alpha':       0.6,
    'legend.facecolor': '#1a1f2e',
    'legend.edgecolor': '#2e3448',
})
print(f"numpy  {np.__version__} | pandas {pd.__version__}")


## Exercice 1 — Operations Matricielles




In [ ]:
# Creation de la matrice 3x3
A = np.array([
    [2, 1, 3],
    [0, 4, 2],
    [1, 0, 5]
], dtype=float)

print("Matrice A :")
print(A)

# --- Determinant ---
det_A = np.linalg.det(A)
print(f"\nDeterminant de A : {det_A:.4f}")

if abs(det_A) > 1e-10:
    print("det(A) != 0 -> la matrice est inversible")
else:
    print("det(A) = 0 -> la matrice est singuliere (pas d'inverse)")

# --- Inverse ---
A_inv = np.linalg.inv(A)
print("\nInverse A⁻¹ :")
print(np.round(A_inv, 4))

# --- Verification : A x A⁻¹ = I ---
identity_check = np.round(A @ A_inv, 6)
print("\nVerification A @ A⁻¹ (doit etre I) :")
print(identity_check)
print(f"Identite confirmee : {np.allclose(A @ A_inv, np.eye(3))}")

# --- Application : resolution de Ax = b ---
b = np.array([1, 2, 3], dtype=float)
x = np.linalg.solve(A, b)
print(f"\nResolution Ax = b avec b={b}")
print(f"Solution x = {np.round(x, 4)}")
print(f"Verification A@x = {np.round(A @ x, 4)} (doit etre {b})")


## Exercice 2 — Analyse Statistique

Calcul des mesures statistiques fondamentales sur un tableau de 50 nombres aleatoires.


In [ ]:
np.random.seed(42)
data = np.random.randint(1, 100, size=50)

print("Tableau de 50 nombres aleatoires :")
print(data)

mean   = np.mean(data)
median = np.median(data)
std    = np.std(data, ddof=1)   # ddof=1 : ecart-type d'echantillon
var    = np.var(data, ddof=1)
q1     = np.percentile(data, 25)
q3     = np.percentile(data, 75)
iqr    = q3 - q1

print("\nStatistiques descriptives :")
print(f"  Moyenne          : {mean:.4f}")
print(f"  Mediane          : {median:.4f}")
print(f"  Ecart-type       : {std:.4f}")
print(f"  Variance         : {var:.4f}")
print(f"  Minimum          : {data.min()}")
print(f"  Maximum          : {data.max()}")
print(f"  Q1 (25%)         : {q1}")
print(f"  Q3 (75%)         : {q3}")
print(f"  IQR (Q3-Q1)      : {iqr}")
print(f"  Asymetrie        : {stats.skew(data):.4f}")
print(f"  Kurtosis         : {stats.kurtosis(data):.4f}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')
axes[0].hist(data, bins=12, color='#4FC3F7', edgecolor='white', alpha=0.85)
axes[0].axvline(mean,   color='#FF8A65', lw=2, linestyle='--', label=f'Moy={mean:.1f}')
axes[0].axvline(median, color='#81C784', lw=2, linestyle=':',  label=f'Med={median:.1f}')
axes[0].set_title('Distribution — 50 nombres aleatoires', fontweight='bold')
axes[0].legend()
axes[1].boxplot(data, patch_artist=True,
                boxprops=dict(facecolor='#4FC3F7', alpha=0.7),
                medianprops=dict(color='#FF8A65', lw=2))
axes[1].set_title('Boite a moustaches', fontweight='bold')
plt.tight_layout(); plt.show()


## Exercice 3 — Manipulation de Dates

Creation d'un tableau NumPy de dates pour janvier 2023 et conversion de format.


In [ ]:
# Creation du tableau de dates (janvier 2023)
dates_jan = np.arange('2023-01-01', '2023-02-01', dtype='datetime64[D]')

print(f"Tableau de dates — Janvier 2023 ({len(dates_jan)} jours) :")
print(dates_jan)

# Conversion en format YYYY/MM/DD (via Pandas)
dates_series  = pd.Series(dates_jan.astype('datetime64[ns]'))
dates_yyyymmdd = dates_series.dt.strftime('%Y/%m/%d')
print("\nFormat YYYY/MM/DD :")
print(dates_yyyymmdd.values[:10], "...")

# Autres formats utiles
formats = {
    'DD-MM-YYYY': '%d-%m-%Y',
    'Jour, DD Mois YYYY': '%A, %d %B %Y',
    'Semaine ISO': lambda d: d.dt.isocalendar().week.values,
}
print("\nFormat DD-MM-YYYY :")
print(dates_series.dt.strftime('%d-%m-%Y').values[:5])

print("\nExtraction des composantes :")
print(f"  Annees uniques  : {np.unique(dates_jan.astype('datetime64[Y]'))}")
print(f"  Mois uniques    : {np.unique(dates_jan.astype('datetime64[M]'))}")
print(f"  Jour de la sem. (lundi=0) : {dates_series.dt.dayofweek.values[:7]}")

# Tri et operations arithmetiques
delta = dates_jan[-1] - dates_jan[0]
print(f"\nDuree totale : {delta} jours")
print(f"Milieu du mois : {dates_jan[len(dates_jan)//2]}")


## Exercice 4 — Manipulation avec NumPy et Pandas

Creation d'un DataFrame avec des nombres aleatoires, selection conditionnelle et agregations.


In [ ]:
np.random.seed(7)
data_dict = {
    'Employe':       [f'EMP_{i:03d}' for i in range(1, 21)],
    'Departement':   np.random.choice(['RH','IT','Finance','Marketing'], 20),
    'Salaire':       np.random.randint(30000, 90000, 20),
    'Performance':   np.round(np.random.uniform(2.0, 5.0, 20), 1),
    'Anciennete_an': np.random.randint(1, 15, 20),
}
df = pd.DataFrame(data_dict)
print("DataFrame (5 premieres lignes) :")
display(df.head())

# --- Selection conditionnelle ---
print("\n1. Employes avec Salaire > 60 000 :")
display(df[df['Salaire'] > 60000][['Employe','Departement','Salaire']])

print("\n2. Employes IT avec Performance >= 4.0 :")
display(df[(df['Departement']=='IT') & (df['Performance'] >= 4.0)])

print("\n3. Salaires entre 40k et 70k :")
display(df[df['Salaire'].between(40000, 70000)][['Employe','Salaire']].head())

# --- Fonctions d'agregation ---
print("\n4. Agregation par departement :")
agg = df.groupby('Departement').agg(
    Nb_employes   = ('Employe', 'count'),
    Salaire_moy   = ('Salaire', 'mean'),
    Salaire_total = ('Salaire', 'sum'),
    Perf_moy      = ('Performance', 'mean'),
    Anciennete_moy= ('Anciennete_an', 'mean'),
).round(2)
display(agg)

print(f"\n5. Salaire total de l'entreprise : {df['Salaire'].sum():,} €")
print(f"   Salaire moyen global           : {df['Salaire'].mean():,.0f} €")
print(f"   Employe le mieux paye          : {df.loc[df['Salaire'].idxmax(), 'Employe']} ({df['Salaire'].max():,} €)")


## Exercice 5 — Representation d'Images avec NumPy

Les images numeriques sont des tableaux NumPy de dimensions (hauteur, largeur) pour le
niveaux de gris, ou (hauteur, largeur, 3) pour RGB (Rouge, Vert, Bleu).
Chaque element est un entier entre 0 (noir) et 255 (blanc/couleur maximale).


In [ ]:
# --- Image 5x5 en niveaux de gris ---
gray_5x5 = np.array([
    [  0,  50, 100, 150, 200],
    [ 50, 100, 150, 200, 250],
    [100, 150, 128, 150, 100],
    [150, 200, 150, 100,  50],
    [200, 250, 100,  50,   0],
], dtype=np.uint8)

print("Image 5x5 niveaux de gris (valeurs 0-255) :")
print(gray_5x5)
print(f"Shape : {gray_5x5.shape}  | dtype : {gray_5x5.dtype}")

# --- Image RGB synthetique 8x8 ---
np.random.seed(0)
rgb_img = np.zeros((8, 8, 3), dtype=np.uint8)
rgb_img[:4, :4, 0] = 200   # rouge top-left
rgb_img[:4, 4:, 1] = 200   # vert top-right
rgb_img[4:, :4, 2] = 200   # bleu bottom-left
rgb_img[4:, 4:, :] = 150   # gris bottom-right

print(f"\nImage RGB 8x8x3 :")
print(f"Shape : {rgb_img.shape}")
print(f"Canal Rouge [0,0] = {rgb_img[0,0,0]}, Vert [0,0] = {rgb_img[0,0,1]}, Bleu [0,0] = {rgb_img[0,0,2]}")

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('#0f1117')

axes[0].imshow(gray_5x5, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, str(gray_5x5[i,j]), ha='center', va='center', fontsize=9,
                     color='white' if gray_5x5[i,j] < 128 else 'black')
axes[0].set_title('Image 5x5 niveaux de gris', fontweight='bold')
plt.colorbar(axes[0].images[0], ax=axes[0])

axes[1].imshow(rgb_img, interpolation='nearest')
axes[1].set_title('Image RGB 8x8', fontweight='bold')

# Gradient 20x20
gradient = np.linspace(0, 255, 20, dtype=np.uint8)
grad_2d  = np.outer(gradient, gradient)
im = axes[2].imshow(grad_2d, cmap='viridis', interpolation='bicubic')
axes[2].set_title('Gradient 20x20 (viridis)', fontweight='bold')
plt.colorbar(im, ax=axes[2])

for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(f"\nPixel central de l'image 5x5 : {gray_5x5[2,2]}")
print(f"Valeur min : {gray_5x5.min()}  max : {gray_5x5.max()}  moy : {gray_5x5.mean():.1f}")


## Exercice 6 — Test d'Hypothese Basique

**Contexte** : test de l'efficacite d'un programme de formation sur la productivite.

- **H0** (hypothese nulle) : la formation n'a pas d'effet (moy_avant = moy_apres)
- **H1** (hypothese alternative) : la formation ameliore la productivite (moy_apres > moy_avant)
- Test utilise : **test T apparie** (memes employes avant/apres)


In [ ]:
np.random.seed(42)

# Donnees de productivite
productivity_before = np.random.normal(loc=50, scale=10, size=30)
productivity_after  = productivity_before + np.random.normal(loc=5, scale=3, size=30)

# --- Statistiques descriptives ---
print("Statistiques descriptives :")
print(f"  Avant — moy={productivity_before.mean():.2f}  std={productivity_before.std():.2f}")
print(f"  Apres — moy={productivity_after.mean():.2f}  std={productivity_after.std():.2f}")
print(f"  Gain moyen : +{(productivity_after - productivity_before).mean():.2f} points")

# --- Test T apparie avec NumPy (calcul manuel) ---
diff     = productivity_after - productivity_before
n        = len(diff)
mean_d   = np.mean(diff)
std_d    = np.std(diff, ddof=1)
se       = std_d / np.sqrt(n)
t_manual = mean_d / se
print(f"\nTest T apparie (calcul NumPy) :")
print(f"  Differences — moy={mean_d:.4f}  std={std_d:.4f}")
print(f"  T-statistique (manuel) : {t_manual:.4f}")

# --- Verification avec SciPy ---
t_stat, p_value = stats.ttest_rel(productivity_after, productivity_before)
print(f"  T-statistique (scipy)  : {t_stat:.4f}")
print(f"  P-value (bilateral)    : {p_value:.6f}")
print(f"  P-value (unilateral)   : {p_value/2:.6f}")

alpha = 0.05
print(f"\nAlpha = {alpha}")
if p_value / 2 < alpha:
    print(f"  -> H0 REJETEE : la formation ameliore significativement la productivite.")
    print(f"     (p={p_value/2:.4f} < {alpha})")
else:
    print(f"  -> H0 non rejetee (p={p_value/2:.4f} >= {alpha})")

# Intervalle de confiance a 95%
ci_margin = stats.t.ppf(0.975, df=n-1) * se
print(f"\nIntervalle de confiance 95% du gain : [{mean_d-ci_margin:.2f}, {mean_d+ci_margin:.2f}]")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f1117')
axes[0].hist(productivity_before, bins=10, alpha=0.65, color='#4FC3F7', label='Avant', edgecolor='white')
axes[0].hist(productivity_after,  bins=10, alpha=0.65, color='#FF8A65', label='Apres', edgecolor='white')
axes[0].axvline(productivity_before.mean(), color='#4FC3F7', lw=2.5, linestyle='--')
axes[0].axvline(productivity_after.mean(),  color='#FF8A65', lw=2.5, linestyle='--')
axes[0].set_title('Distribution avant / apres formation', fontweight='bold')
axes[0].legend()
axes[1].scatter(range(n), diff, color='#81C784', alpha=0.75, edgecolors='white', s=50)
axes[1].axhline(0,      color='white',   lw=1.5, linestyle='--', alpha=0.5, label='Pas de changement')
axes[1].axhline(mean_d, color='#FFD54F', lw=2,   linestyle='-',  label=f'Gain moy={mean_d:.2f}')
axes[1].fill_between(range(n), mean_d-ci_margin, mean_d+ci_margin,
                     alpha=0.15, color='#FFD54F', label='IC 95%')
axes[1].set_title(f'Gain individuel  (T={t_stat:.2f}, p={p_value:.4f})', fontweight='bold')
axes[1].set_xlabel('Employe'); axes[1].set_ylabel('Gain de productivite')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## Exercice 7 — Comparaison Complexe de Tableaux

Comparaison element par element de deux tableaux NumPy.


In [ ]:
np.random.seed(10)
arr1 = np.random.randint(1, 20, size=10)
arr2 = np.random.randint(1, 20, size=10)

print(f"arr1 : {arr1}")
print(f"arr2 : {arr2}")

# Comparaisons element par element
greater  = arr1 > arr2
lesser   = arr1 < arr2
equal    = arr1 == arr2

print(f"\narr1 > arr2 : {greater}")
print(f"arr1 < arr2 : {lesser}")
print(f"arr1 = arr2 : {equal}")

# Statistiques
print(f"\nNombre de positions ou arr1 > arr2 : {greater.sum()}")
print(f"Nombre de positions ou arr1 < arr2 : {lesser.sum()}")
print(f"Nombre de positions ou arr1 = arr2 : {equal.sum()}")

# Indices
print(f"\nIndices ou arr1 > arr2 : {np.where(greater)[0]}")
print(f"Valeurs de arr1 > arr2 : {arr1[greater]}")

# Operations avancees
diff = arr1.astype(int) - arr2.astype(int)
print(f"\nDifferences arr1 - arr2 : {diff}")
print(f"Maximum element-wise    : {np.maximum(arr1, arr2)}")
print(f"Minimum element-wise    : {np.minimum(arr1, arr2)}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0f1117')
x = np.arange(len(arr1))
axes[0].bar(x-0.2, arr1, 0.35, label='arr1', color='#4FC3F7', edgecolor='white', alpha=0.85)
axes[0].bar(x+0.2, arr2, 0.35, label='arr2', color='#FF8A65', edgecolor='white', alpha=0.85)
for i in range(len(arr1)):
    if arr1[i] > arr2[i]:
        axes[0].annotate('>', (i, max(arr1[i],arr2[i])+0.3), ha='center', color='#FFD54F', fontweight='bold')
axes[0].set_title('Comparaison element-wise', fontweight='bold')
axes[0].legend()
colors_bool = ['#81C784' if g else '#FF8A65' for g in greater]
axes[1].bar(x, greater.astype(int), color=colors_bool, edgecolor='white', alpha=0.88)
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['arr2 >= arr1','arr1 > arr2'])
axes[1].set_title('Masque booleen arr1 > arr2', fontweight='bold')
plt.tight_layout(); plt.show()


## Exercice 8 — Manipulation de Series Temporelles

Generation et decoupage de donnees temporelles pour 2023.


In [ ]:
np.random.seed(0)

# Generation : serie temporelle journaliere sur 2023
dates_2023 = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')
n_days     = len(dates_2023)

# Donnees synthetiques : temperature + saisonnalite
angle     = np.linspace(0, 2*np.pi, n_days)
seasonal  = 15 * np.sin(angle - np.pi/2)
noise     = np.random.normal(0, 3, n_days)
temp_2023 = (15 + seasonal + noise).round(1)

ts = pd.Series(temp_2023, index=dates_2023, name='Temperature_C')
print(f"Serie temporelle 2023 : {len(ts)} jours")
print(ts.head())

# --- Decoupages trimestriels ---
q1 = ts['2023-01':'2023-03']   # Janvier - Mars
q2 = ts['2023-04':'2023-06']   # Avril   - Juin
q3 = ts['2023-07':'2023-09']   # Juillet - Septembre
q4 = ts['2023-10':'2023-12']   # Octobre - Decembre

quarters = {'Jan-Mar (Q1)': q1, 'Avr-Jun (Q2)': q2,
            'Jul-Sep (Q3)': q3, 'Oct-Dec (Q4)': q4}
print("\nMoyenne par trimestre :")
for name, q in quarters.items():
    print(f"  {name} : {q.mean():.2f}°C  ({len(q)} jours)")

# Resampling mensuel
monthly_mean = ts.resample('ME').mean().round(2)
print("\nMoyenne mensuelle :")
print(monthly_mean)

# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
fig.patch.set_facecolor('#0f1117')
colors_q = ['#4FC3F7','#81C784','#FF8A65','#FFD54F']
for (name, q), color in zip(quarters.items(), colors_q):
    axes[0].plot(q.index, q.values, color=color, lw=1.5, label=name)
axes[0].axhline(ts.mean(), color='white', lw=1.5, linestyle='--',
                alpha=0.6, label=f'Moy. annuelle={ts.mean():.1f}°C')
axes[0].set_title('Serie temporelle 2023 — decoupages trimestriels', fontweight='bold')
axes[0].set_ylabel('Temperature (°C)'); axes[0].legend(fontsize=9)

axes[1].bar(monthly_mean.index, monthly_mean.values,
            color=colors_q * 3, edgecolor='#0f1117', alpha=0.88, width=20)
for i, (dt, val) in enumerate(monthly_mean.items()):
    axes[1].text(dt, val+0.3, f'{val:.1f}', ha='center', fontsize=8, color='#e0e0e0')
axes[1].set_title('Moyenne mensuelle', fontweight='bold')
axes[1].set_ylabel('Temperature (°C)')
plt.tight_layout(); plt.show()


## Exercice 9 — Conversion NumPy <-> Pandas


In [ ]:
np.random.seed(3)

# --- NumPy -> Pandas ---
array_np = np.random.randint(10, 100, size=(5, 4))
print("Tableau NumPy (5x4) :")
print(array_np)
print(f"Type : {type(array_np)}  |  Shape : {array_np.shape}  |  dtype : {array_np.dtype}")

# Methode 1 : pd.DataFrame directement
df1 = pd.DataFrame(array_np, columns=['A','B','C','D'])
print("\nDataFrame (sans index custom) :")
display(df1)

# Methode 2 : avec index personnalise
df2 = pd.DataFrame(array_np,
                   index   = [f'Row_{i}' for i in range(1,6)],
                   columns = ['Score1','Score2','Score3','Score4'])
print("DataFrame (avec index custom) :")
display(df2)

# Series a partir d'un tableau 1D
arr_1d   = np.array([10, 20, 30, 40, 50])
series_s = pd.Series(arr_1d, index=['a','b','c','d','e'], name='Values')
print(f"Series depuis tableau 1D :\n{series_s}")

# --- Pandas -> NumPy ---
back_to_numpy = df2.to_numpy()
print(f"\nRetour vers NumPy :")
print(back_to_numpy)
print(f"Type : {type(back_to_numpy)}  |  Shape : {back_to_numpy.shape}")

# Verification que les valeurs sont identiques
print(f"\nValeurs identiques : {np.array_equal(array_np, back_to_numpy)}")

# Operations NumPy sur le DataFrame
print(f"\nMoyenne des colonnes (NumPy sur df.values) : {np.mean(df2.values, axis=0).round(1)}")
print(f"Ecart-type global                           : {np.std(df2.values):.2f}")


## Exercice 10 — Visualisation de Base avec Matplotlib


In [ ]:
np.random.seed(42)

# Donnees
x  = np.linspace(0, 4*np.pi, 200)
y1 = np.sin(x)
y2 = np.cos(x)
y3 = np.sin(x) * np.exp(-x/10)   # sinusoide amortie
y_rand = np.cumsum(np.random.normal(0, 0.3, 200))  # marche aleatoire

fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#0f1117')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

# ── 1. Graphique lineaire : sin, cos, amorti ─────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(x, y1, color='#4FC3F7', lw=2,   label='sin(x)')
ax1.plot(x, y2, color='#FF8A65', lw=2,   label='cos(x)')
ax1.plot(x, y3, color='#81C784', lw=2.5, label='sin(x)·e^{-x/10}', linestyle='--')
ax1.axhline(0, color='white', lw=0.8, linestyle=':', alpha=0.5)
ax1.set_title('Fonctions trigonometriques', fontweight='bold')
ax1.set_xlabel('x (radians)'); ax1.set_ylabel('y')
ax1.legend(fontsize=9)

# ── 2. Marche aleatoire ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(y_rand, color='#CE93D8', lw=1.8, alpha=0.9)
ax2.fill_between(range(len(y_rand)), y_rand, alpha=0.15, color='#CE93D8')
ax2.axhline(0, color='white', lw=1, linestyle='--', alpha=0.5)
ax2.set_title('Marche aleatoire (200 pas)', fontweight='bold')
ax2.set_xlabel('Etape'); ax2.set_ylabel('Position')

# ── 3. Histogramme d'une distribution normale ────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
data_norm = np.random.normal(0, 1, 1000)
ax3.hist(data_norm, bins=35, color='#FFD54F', edgecolor='#0f1117', alpha=0.85, density=True)
x_curve = np.linspace(-4, 4, 200)
ax3.plot(x_curve, (1/np.sqrt(2*np.pi))*np.exp(-x_curve**2/2),
         color='#FF8A65', lw=2.5, label='N(0,1) theorique')
ax3.axvline(data_norm.mean(), color='white', lw=2, linestyle='--',
            label=f'Moy={data_norm.mean():.2f}')
ax3.set_title('Distribution normale N(0,1) — 1000 echantillons', fontweight='bold')
ax3.set_xlabel('Valeur'); ax3.set_ylabel('Densite')
ax3.legend(fontsize=9)

# ── 4. Scatter plot : relation quadratique + bruit ────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
x_sc   = np.linspace(-3, 3, 100)
y_sc   = x_sc**2 + np.random.normal(0, 0.8, 100)
colors = plt.cm.plasma(np.linspace(0, 1, 100))
ax4.scatter(x_sc, y_sc, c=colors, s=40, alpha=0.8, edgecolors='none')
x_fit  = np.linspace(-3, 3, 200)
ax4.plot(x_fit, x_fit**2, color='white', lw=2.5, linestyle='--', label='y = x² (theorique)')
ax4.set_title('Scatter : y = x² + bruit gaussien', fontweight='bold')
ax4.set_xlabel('x'); ax4.set_ylabel('y')
ax4.legend(fontsize=9)

fig.suptitle('Exercice 10 — Visualisations Matplotlib avec NumPy',
             color='#e0e0e0', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('matplotlib_basics.png', dpi=140, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("4 graphiques generes avec succes")
